# Gradient Boosting and XGBoost

A comprehensive guide to gradient boosting algorithms, the most powerful techniques for structured/tabular data.

## Learning Objectives

- Understand boosting vs bagging ensemble methods
- Master gradient boosting concepts and math
- Implement Gradient Boosting with scikit-learn
- Learn XGBoost, LightGBM, and CatBoost
- Tune hyperparameters for best performance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_regression, load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Boosting vs Bagging

**Bagging (Bootstrap Aggregating):**
- Train models independently on random subsets (parallel)
- Average predictions to reduce variance
- Example: Random Forest

**Boosting:**
- Train models sequentially (serial)
- Each model corrects errors of previous models
- Reduces bias and variance
- Examples: AdaBoost, Gradient Boosting, XGBoost

In [ ]:
# Visual comparison of Bagging vs Boosting concept
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bagging illustration
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_title('Bagging (Random Forest)', fontsize=14)
ax.axis('off')

# Data box
ax.add_patch(plt.Rectangle((1, 6), 8, 1.5, fill=True, color='lightblue', ec='black'))
ax.text(5, 6.75, 'Full Dataset', ha='center', va='center', fontsize=12)

# Bootstrap samples
for i, x in enumerate([1.5, 4, 6.5]):
    ax.annotate('', xy=(x+0.5, 5), xytext=(x+0.5, 6),
                arrowprops=dict(arrowstyle='->', color='gray'))
    ax.add_patch(plt.Rectangle((x, 3.5), 2, 1.5, fill=True, color='lightyellow', ec='black'))
    ax.text(x+1, 4.25, f'Sample {i+1}', ha='center', va='center', fontsize=10)

# Models
for i, x in enumerate([1.5, 4, 6.5]):
    ax.annotate('', xy=(x+0.5, 2.3), xytext=(x+0.5, 3.5),
                arrowprops=dict(arrowstyle='->', color='gray'))
    ax.add_patch(plt.Rectangle((x, 1), 2, 1.3, fill=True, color='lightgreen', ec='black'))
    ax.text(x+1, 1.65, f'Tree {i+1}', ha='center', va='center', fontsize=10)

# Voting
ax.text(5, 0.3, 'Average/Vote → Final Prediction', ha='center', fontsize=12, style='italic')

# Boosting illustration
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_title('Boosting (Gradient Boosting)', fontsize=14)
ax.axis('off')

# Sequential models
y_positions = [6.5, 4.5, 2.5]
colors = ['lightblue', 'lightyellow', 'lightgreen']
labels = ['Model 1: Fit data', 'Model 2: Fit residuals', 'Model 3: Fit residuals']

for i, (y, color, label) in enumerate(zip(y_positions, colors, labels)):
    ax.add_patch(plt.Rectangle((1, y), 6, 1.3, fill=True, color=color, ec='black'))
    ax.text(4, y+0.65, label, ha='center', va='center', fontsize=11)
    if i < 2:
        ax.annotate('', xy=(4, y), xytext=(4, y-0.7),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.text(5, 0.8, 'Sum weighted predictions → Final', ha='center', fontsize=12, style='italic')

plt.tight_layout()
plt.show()

## 2. Gradient Boosting Algorithm

**Core Idea:** Fit new models to the residual errors of previous models.

**Algorithm:**
1. Initialize with a constant prediction (e.g., mean)
2. For m = 1 to M:
   - Compute pseudo-residuals (negative gradient of loss)
   - Fit a weak learner to pseudo-residuals
   - Update model: F_m(x) = F_{m-1}(x) + η * h_m(x)
3. Final prediction: F(x) = Σ η * h_m(x)

In [ ]:
# Simple gradient boosting from scratch (regression)
from sklearn.tree import DecisionTreeRegressor

# Generate simple data
np.random.seed(42)
X_simple = np.linspace(0, 10, 100).reshape(-1, 1)
y_simple = np.sin(X_simple.ravel()) + np.random.randn(100) * 0.2

class SimpleGradientBoosting:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.initial_prediction = None
        
    def fit(self, X, y):
        # Initialize with mean
        self.initial_prediction = np.mean(y)
        
        # Current predictions
        F = np.full(len(y), self.initial_prediction)
        
        for i in range(self.n_estimators):
            # Compute residuals (negative gradient for MSE loss)
            residuals = y - F
            
            # Fit tree to residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            
            # Update predictions
            F += self.learning_rate * tree.predict(X)
            
        return self
    
    def predict(self, X):
        F = np.full(len(X), self.initial_prediction)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F

# Visualize boosting iterations
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

n_estimators_list = [1, 2, 5, 10, 20, 50]

for ax, n_est in zip(axes, n_estimators_list):
    gb = SimpleGradientBoosting(n_estimators=n_est, learning_rate=0.3, max_depth=2)
    gb.fit(X_simple, y_simple)
    y_pred = gb.predict(X_simple)
    
    ax.scatter(X_simple, y_simple, alpha=0.5, s=20, label='Data')
    ax.plot(X_simple, y_pred, 'r-', linewidth=2, label='Prediction')
    ax.set_title(f'{n_est} Trees (MSE: {mean_squared_error(y_simple, y_pred):.3f})')
    ax.legend()

plt.suptitle('Gradient Boosting: Effect of Number of Trees', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Scikit-learn Gradient Boosting

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)

# Evaluate
y_pred = gb_clf.predict(X_test)
print("Gradient Boosting Classifier Results:")
print(f"Training Accuracy: {gb_clf.score(X_train, y_train):.3f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': cancer.feature_names,
    'importance': gb_clf.feature_importances_
}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 features
top_features = feature_importance.head(10)
axes[0].barh(top_features['feature'], top_features['importance'], color='steelblue')
axes[0].set_xlabel('Importance')
axes[0].set_title('Top 10 Feature Importances')
axes[0].invert_yaxis()

# Training history (staged predictions)
train_scores = []
test_scores = []

for i, y_pred_train in enumerate(gb_clf.staged_predict(X_train)):
    train_scores.append(accuracy_score(y_train, y_pred_train))
    
for i, y_pred_test in enumerate(gb_clf.staged_predict(X_test)):
    test_scores.append(accuracy_score(y_test, y_pred_test))

axes[1].plot(range(1, len(train_scores) + 1), train_scores, 'b-', label='Train')
axes[1].plot(range(1, len(test_scores) + 1), test_scores, 'r-', label='Test')
axes[1].set_xlabel('Number of Trees')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Progress')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Key Hyperparameters

| Parameter | Description | Effect |
|-----------|-------------|--------|
| `n_estimators` | Number of boosting stages | More = better fit, risk overfitting |
| `learning_rate` | Shrinks contribution of each tree | Lower = needs more trees, but generalizes better |
| `max_depth` | Maximum depth of trees | Higher = more complex, risk overfitting |
| `min_samples_split` | Min samples to split node | Higher = more regularization |
| `subsample` | Fraction of samples per tree | < 1.0 adds stochasticity (like Random Forest) |

In [ ]:
# Effect of learning rate and n_estimators trade-off
learning_rates = [0.01, 0.1, 0.5, 1.0]
n_estimators_range = range(10, 201, 10)

fig, ax = plt.subplots(figsize=(10, 6))

for lr in learning_rates:
    test_scores = []
    for n_est in n_estimators_range:
        gb = GradientBoostingClassifier(
            n_estimators=n_est,
            learning_rate=lr,
            max_depth=3,
            random_state=42
        )
        gb.fit(X_train, y_train)
        test_scores.append(gb.score(X_test, y_test))
    
    ax.plot(n_estimators_range, test_scores, label=f'lr={lr}')

ax.set_xlabel('Number of Estimators')
ax.set_ylabel('Test Accuracy')
ax.set_title('Learning Rate vs Number of Estimators Trade-off')
ax.legend()
ax.set_ylim(0.9, 1.0)
plt.show()

## 5. XGBoost

XGBoost (eXtreme Gradient Boosting) is an optimized implementation with:
- Regularization (L1 and L2)
- Parallel processing
- Tree pruning
- Built-in handling of missing values
- Cross-validation built-in

In [ ]:
# Install XGBoost if needed
try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    xgb = None

In [ ]:
if xgb:
    # XGBoost Classifier
    xgb_clf = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        reg_lambda=1,  # L2 regularization
        reg_alpha=0,   # L1 regularization
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    # Train with early stopping
    xgb_clf.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    y_pred = xgb_clf.predict(X_test)
    print("XGBoost Classifier Results:")
    print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")

In [ ]:
if xgb:
    # XGBoost feature importance
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Built-in feature importance plot
    xgb.plot_importance(xgb_clf, ax=axes[0], max_num_features=10, importance_type='gain')
    axes[0].set_title('XGBoost Feature Importance (Gain)')
    
    # Compare importance types
    importance_types = ['weight', 'gain', 'cover']
    importance_df = pd.DataFrame()
    
    for imp_type in importance_types:
        importance = xgb_clf.get_booster().get_score(importance_type=imp_type)
        importance_df[imp_type] = pd.Series(importance)
    
    importance_df = importance_df.fillna(0)
    importance_df = importance_df.div(importance_df.sum())  # Normalize
    
    importance_df.head(10).plot(kind='bar', ax=axes[1])
    axes[1].set_title('Feature Importance by Different Metrics')
    axes[1].set_xlabel('Feature')
    axes[1].set_ylabel('Normalized Importance')
    axes[1].legend(title='Metric')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

## 6. LightGBM

LightGBM is another fast gradient boosting library with:
- Histogram-based learning (faster)
- Leaf-wise tree growth (vs level-wise)
- Lower memory usage
- Better for large datasets

In [ ]:
try:
    import lightgbm as lgb
    print(f"LightGBM version: {lgb.__version__}")
    
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        num_leaves=31,
        random_state=42,
        verbose=-1
    )
    lgb_clf.fit(X_train, y_train)
    
    y_pred = lgb_clf.predict(X_test)
    print(f"LightGBM Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    
except ImportError:
    print("LightGBM not installed. Run: pip install lightgbm")
    lgb = None

## 7. Hyperparameter Tuning

In [ ]:
# Grid search for Gradient Boosting
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(gb, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_:.3f}")
print(f"Test Score: {grid_search.score(X_test, y_test):.3f}")

In [ ]:
# Visualize grid search results
results = pd.DataFrame(grid_search.cv_results_)

# Heatmap for learning_rate vs n_estimators
pivot = results.pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_n_estimators',
    aggfunc='mean'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Grid Search Results: Learning Rate vs N_Estimators')
plt.show()

## 8. Gradient Boosting for Regression

In [ ]:
# Generate regression data
X_reg, y_reg = make_regression(n_samples=1000, n_features=10, noise=20, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Train Gradient Boosting Regressor
gb_reg = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    loss='squared_error',
    random_state=42
)
gb_reg.fit(X_train_reg, y_train_reg)

y_pred_reg = gb_reg.predict(X_test_reg)

print("Gradient Boosting Regressor Results:")
print(f"R² Score: {r2_score(y_test_reg, y_pred_reg):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.2f}")

In [ ]:
# Residual analysis
residuals = y_test_reg - y_pred_reg

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Predicted vs Actual
axes[0].scatter(y_test_reg, y_pred_reg, alpha=0.5)
axes[0].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Predicted vs Actual')

# Residuals distribution
axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residuals Distribution')

# Residuals vs Predicted
axes[2].scatter(y_pred_reg, residuals, alpha=0.5)
axes[2].axhline(0, color='r', linestyle='--')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

## 9. Model Comparison

In [ ]:
import time

# Compare different boosting and ensemble methods
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

if xgb:
    models['XGBoost'] = xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
    
if lgb:
    models['LightGBM'] = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)

results = []
for name, model in models.items():
    start_time = time.time()
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    elapsed = time.time() - start_time
    
    results.append({
        'Model': name,
        'Mean Accuracy': scores.mean(),
        'Std': scores.std(),
        'Time (s)': elapsed
    })

results_df = pd.DataFrame(results).sort_values('Mean Accuracy', ascending=False)
print("Model Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy comparison
axes[0].barh(results_df['Model'], results_df['Mean Accuracy'], xerr=results_df['Std'], color='steelblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xlim(0.9, 1.0)

# Training time comparison
axes[1].barh(results_df['Model'], results_df['Time (s)'], color='coral')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Training Time Comparison')

plt.tight_layout()
plt.show()

## 10. Best Practices for Gradient Boosting

**Hyperparameter Tuning Strategy:**
1. Start with default parameters
2. Set `n_estimators` high, use early stopping
3. Tune `max_depth` and `min_samples_split` first
4. Then tune `learning_rate` and `n_estimators` together
5. Finally tune `subsample` and `colsample_bytree`

**Common Pitfalls:**
- Overfitting with too many trees/high learning rate
- Not using early stopping
- Ignoring feature scaling (less critical for trees, but helps XGBoost/LightGBM)
- Not handling categorical variables properly

In [ ]:
# Early stopping example with XGBoost
if xgb:
    xgb_early = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        early_stopping_rounds=20,
        eval_metric='logloss'
    )
    
    xgb_early.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    print(f"Best iteration: {xgb_early.best_iteration}")
    print(f"Test Accuracy: {xgb_early.score(X_test, y_test):.3f}")

## 11. Key Takeaways

1. **Gradient Boosting** builds trees sequentially, each correcting previous errors
2. **Learning rate** and **n_estimators** have an inverse relationship
3. **XGBoost** and **LightGBM** are optimized implementations for production
4. Use **early stopping** to prevent overfitting
5. **Feature importance** is built-in and reliable
6. Gradient boosting typically outperforms Random Forest on tabular data
7. **Trade-off**: More powerful but harder to tune than Random Forest

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    'Algorithm': ['Random Forest', 'AdaBoost', 'Gradient Boosting', 'XGBoost', 'LightGBM'],
    'Type': ['Bagging', 'Boosting', 'Boosting', 'Boosting', 'Boosting'],
    'Speed': ['Fast', 'Moderate', 'Slow', 'Fast', 'Fastest'],
    'Accuracy': ['Good', 'Good', 'Very Good', 'Excellent', 'Excellent'],
    'Regularization': ['None', 'None', 'None', 'L1/L2', 'L1/L2'],
    'Missing Values': ['No', 'No', 'No', 'Yes', 'Yes'],
    'Best For': ['Quick baseline', 'Weak learners', 'Small data', 'Competitions', 'Large data']
})

print("Gradient Boosting Algorithms Comparison:")
print(comparison.to_string(index=False))